In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns


df = pd.read_csv("../data/processed/final_dataset.csv", index_col=0, parse_dates=True)
print("Dataset loaded. Shape:", df.shape)

df.head()


target = "target_cpi_next_month"

X = df.drop(columns=[target])
y = df[target]

print("Number of features:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # time series → no shuffle
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained successfully.")

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model performance:")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test, label="Actual CPI", linewidth=2)
plt.plot(y_test.index, y_pred, label="Predicted CPI", linewidth=2)
plt.title("Actual vs Predicted CPI (Next Month)")
plt.xlabel("Date")
plt.ylabel("CPI")
plt.legend()
plt.grid(True)
plt.show()

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importances.head(20), x="importance", y="feature")
plt.title("Top 20 Most Important Features")
plt.show()

importances.head(20)

print("Model saved to data/processed/random_forest_model.pkl")